# 05.07 - Image I/O for CV

**Daily output:** a reliable `safe_image_loader()` function.

Today is about the quiet part of CV that can silently ruin a model: image loading. You will practice PIL vs OpenCV, RGB/BGR, grayscale and alpha channels, dtype and value range, resizing, aspect ratio, corrupt file handling, and the final HWC/CHW shape contract.

Masterplan resources: Pillow Handbook, OpenCV Tutorials, Kaggle Computer Vision.

**Notebook type:** Solution notebook with full working code.


## Mental Model

PIL/Pillow loads images as `PIL.Image.Image` objects. For color images, `img.convert("RGB")` gives predictable RGB data, and `np.array(img)` gives `[H, W, C]`.

OpenCV loads images as NumPy arrays. `cv2.imread(path)` returns BGR by default and returns `None` if reading fails. Before plotting or feeding a model that expects RGB, convert with `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`.

A strong contest pipeline forces a contract:

- color: RGB
- image array while inspecting: `[H, W, C]`
- tensor before PyTorch model: `[C, H, W]`
- dtype before model: `float32`
- value range before model: usually `[0, 1]`, then optional mean/std normalization


In [ ]:
from pathlib import Path
import random

import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError

try:
    import cv2
except ImportError:
    cv2 = None
    print("OpenCV is not installed. OpenCV examples will be skipped.")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed. Visualization examples will be skipped.")

random.seed(42)
np.random.seed(42)

ROOT = Path("_day05_demo_images")
ROOT.mkdir(exist_ok=True)


In [ ]:
def make_demo_images(root=ROOT):
    h, w = 96, 128
    y = np.linspace(0, 255, h, dtype=np.uint8)[:, None]
    x = np.linspace(0, 255, w, dtype=np.uint8)[None, :]

    red = np.repeat(x, h, axis=0)
    green = np.repeat(y, w, axis=1)
    blue = np.full((h, w), 80, dtype=np.uint8)
    rgb = np.stack([red, green, blue], axis=-1)
    Image.fromarray(rgb, mode="RGB").save(root / "rgb_gradient.png")

    checker = (((np.indices((80, 80)).sum(axis=0) // 10) % 2) * 255).astype(np.uint8)
    checker_rgb = np.stack([checker, np.zeros_like(checker), 255 - checker], axis=-1)
    Image.fromarray(checker_rgb, mode="RGB").save(root / "checker.jpg", quality=95)

    gray = np.tile(np.linspace(0, 255, 100, dtype=np.uint8), (60, 1))
    Image.fromarray(gray, mode="L").save(root / "grayscale.png")

    rgba = np.zeros((70, 110, 4), dtype=np.uint8)
    rgba[..., 0] = 255
    rgba[..., 1] = 180
    rgba[..., 3] = np.linspace(30, 255, 110, dtype=np.uint8)[None, :]
    Image.fromarray(rgba, mode="RGBA").save(root / "rgba.png")

    (root / "corrupt.jpg").write_bytes(b"this is not a valid image file")

make_demo_images()
sorted(p.name for p in ROOT.iterdir())


## Inspect Image Data

When debugging input, print mode, shape, dtype, min/max, and a sample pixel. Most image bugs become obvious after those checks.


In [ ]:
def describe_array(name, arr):
    if arr is None:
        print(f"{name}: None")
        return
    first = arr.reshape(-1, arr.shape[-1] if arr.ndim == 3 else 1)[0]
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, min={arr.min()}, max={arr.max()}, first_pixel={first}")

image_path = ROOT / "rgb_gradient.png"

pil_img = Image.open(image_path)
pil_arr = np.array(pil_img)
print("PIL mode:", pil_img.mode)
describe_array("PIL array", pil_arr)

if cv2 is not None:
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    describe_array("OpenCV raw BGR", bgr)
    describe_array("OpenCV converted RGB", rgb)
    print("PIL equals OpenCV-converted RGB:", np.array_equal(pil_arr, rgb))


## RGB vs BGR

If a red object appears blue, suspect channel order. OpenCV's BGR output must be converted before plotting or mixing with PIL/torchvision-style RGB transforms.


In [ ]:
if cv2 is not None and plt is not None:
    path = ROOT / "checker.jpg"
    bgr = cv2.imread(str(path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(bgr)
    axes[0].set_title("Wrong: BGR shown as RGB")
    axes[1].imshow(rgb)
    axes[1].set_title("Correct: converted RGB")
    for ax in axes:
        ax.axis("off")
    plt.show()
else:
    print("Install cv2 and matplotlib to view this comparison.")


## Grayscale and Alpha Channels

Real datasets can contain grayscale (`L`) or transparent (`RGBA`) images. Use `convert("RGB")` when the model expects exactly three channels.


In [ ]:
for path in [ROOT / "grayscale.png", ROOT / "rgba.png"]:
    img = Image.open(path)
    arr = np.array(img)
    print(path.name, "mode=", img.mode, "shape=", arr.shape)

    rgb = img.convert("RGB")
    print(" after convert('RGB'):", np.array(rgb).shape)


## Resizing Choices

Direct resize is fast and simple, but it can distort aspect ratio. Aspect-ratio resize plus padding preserves geometry but adds borders. For quick baselines, direct resize is often acceptable; for objects, documents, video frames, or medical-like data, padding is often safer.


In [ ]:
def resize_stretch(img, size=(224, 224)):
    return img.resize(size, resample=Image.BILINEAR)

def resize_keep_aspect_and_pad(img, size=(224, 224), fill=(0, 0, 0)):
    img = img.convert("RGB")
    img.thumbnail(size, resample=Image.BILINEAR)
    canvas = Image.new("RGB", size, fill)
    left = (size[0] - img.width) // 2
    top = (size[1] - img.height) // 2
    canvas.paste(img, (left, top))
    return canvas

sample = Image.open(ROOT / "rgb_gradient.png").convert("RGB")
print("original:", sample.size)
print("stretch:", resize_stretch(sample).size)
print("pad:", resize_keep_aspect_and_pad(sample).size)


## Daily Output: `safe_image_loader()`

This loader catches missing/corrupt files, fixes EXIF orientation, converts to RGB, optionally resizes, and returns either PIL, HWC uint8, or CHW float32.


In [ ]:
def safe_image_loader(path, size=None, keep_aspect=False, output="pil", strict=False, fill=(0, 0, 0)):
    path = Path(path)
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")

            if size is not None:
                if keep_aspect:
                    img = resize_keep_aspect_and_pad(img, size=size, fill=fill)
                else:
                    img = img.resize(size, resample=Image.BILINEAR)

            if output == "pil":
                return img.copy()

            arr = np.array(img)
            if output == "hwc_uint8":
                return arr
            if output == "chw_float32":
                arr = arr.astype(np.float32) / 255.0
                return np.transpose(arr, (2, 0, 1))
            if output == "bgr_uint8":
                return arr[..., ::-1].copy()

            raise ValueError(f"Unknown output={output!r}")

    except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError) as exc:
        if strict:
            raise
        print(f"Warning: failed to load {path}: {exc}")
        return None

good = safe_image_loader(ROOT / "rgba.png", size=(64, 64), output="chw_float32")
bad = safe_image_loader(ROOT / "corrupt.jpg", size=(64, 64), output="chw_float32")

print("good:", good.shape, good.dtype, float(good.min()), float(good.max()))
print("bad:", bad)


## Folder Scan

Before training, scan the folder once. Count good files, bad files, image modes, and dimensions. This catches bad paths and weird channels early.


In [ ]:
def scan_image_folder(root):
    root = Path(root)
    rows = []
    for path in sorted(root.iterdir()):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            continue
        try:
            with Image.open(path) as img:
                rows.append({"path": str(path), "ok": True, "mode": img.mode, "width": img.width, "height": img.height, "error": ""})
        except Exception as exc:
            rows.append({"path": str(path), "ok": False, "mode": "", "width": None, "height": None, "error": type(exc).__name__})
    return rows

for row in scan_image_folder(ROOT):
    print(row)


## Day 05 Checklist

Verify path exists, image decodes, color is RGB, shape is expected, dtype is expected, value range is expected, train/validation preprocessing is intentional, and corrupt files have a defined policy: skip, placeholder, or fail fast.
